In [3]:
!pip install pandas numpy scikit-learn xgboost lightgbm catboost matplotlib category_encoders pygeohash optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.7 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import r2_score
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
import matplotlib.pyplot as plt
import pygeohash as pgh
import optuna

SEED = 42
np.random.seed(SEED)
print('Libraries loaded ✓')

# ── Load Data & Clean Nulls ──
train_raw = pd.read_csv('train.csv')
test_raw  = pd.read_csv('test.csv')

print(f'train: {train_raw.shape}  test: {test_raw.shape}')

def clean_nulls(df):
    df = df.copy()
    
    # 1. Fill Categorical Nulls
    for col in ['RoadType','Weather','LargeVehicles','Landmarks','geohash']:
        if col in df.columns: df[col] = df[col].fillna('Unknown')
            
    # 2. Fill Numeric Nulls (WITH STRING CONVERSION FIX)
    for col in ['Temperature','NumberofLanes']:
        if col in df.columns: 
            # Force the column to be a number (turns text like '13.2' into float 13.2)
            df[col] = pd.to_numeric(df[col], errors='coerce')
            # Now the median math will work perfectly!
            df[col] = df[col].fillna(df[col].median())
            
    # 3. Fill Timestamp Nulls
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].fillna('0:0')
        
    return df

train_raw = clean_nulls(train_raw)
test_raw  = clean_nulls(test_raw)
print("Data loaded and cleaned successfully!")

Libraries loaded ✓
train: (119078, 11)  test: (41778, 10)
Data loaded and cleaned successfully!


In [12]:
print("--- 🚀 ENGINEERING CLEAN MEGA FEATURES ---")
def engineer_mega_features(train_raw, test_raw):
    train = train_raw.copy()
    test = test_raw.copy()
    train['is_test'] = 0
    test['is_test'] = 1
    
    df_all = pd.concat([train, test], ignore_index=True)
    
    def parse_h(ts):
        try: return int(str(ts).split(':')[0])
        except: return 0
    def parse_m(ts):
        try: return int(str(ts).split(':')[1])
        except: return 0
        
    df_all['hour'] = df_all['timestamp'].apply(parse_h)
    df_all['minute'] = df_all['timestamp'].apply(parse_m)
    df_all['hour_sin'] = np.sin(2 * np.pi * df_all['hour'] / 24)
    df_all['hour_cos'] = np.cos(2 * np.pi * df_all['hour'] / 24)
    df_all['rush_hour'] = df_all['hour'].apply(lambda x: 1 if (7 <= x <= 10) or (17 <= x <= 20) else 0)
    
    # ── THE FIX: Safe Geohash Decoder ──
    def safe_lat(x):
        try:
            if x == 'Unknown' or pd.isna(x): return np.nan
            return pgh.decode(str(x).strip())[0]
        except:
            return np.nan

    def safe_lon(x):
        try:
            if x == 'Unknown' or pd.isna(x): return np.nan
            return pgh.decode(str(x).strip())[1]
        except:
            return np.nan

    df_all['latitude'] = df_all['geohash'].apply(safe_lat)
    df_all['longitude'] = df_all['geohash'].apply(safe_lon)
    # ───────────────────────────────────

    mean_lat, mean_lon = df_all['latitude'].mean(), df_all['longitude'].mean()
    df_all['dist_from_center'] = np.sqrt((df_all['latitude'] - mean_lat)**2 + (df_all['longitude'] - mean_lon)**2)
    
    # Safely get prefixes (handling floats/nans)
    df_all['geo_prefix4'] = df_all['geohash'].astype(str).str[:4]
    df_all['geo_prefix5'] = df_all['geohash'].astype(str).str[:5]
    
    df_all['lv_allowed'] = (df_all['LargeVehicles'] == 'Allowed').astype(int)
    df_all['lane_lv_ratio'] = df_all['NumberofLanes'] / (df_all['lv_allowed'] + 1)
    df_all['Road_LV'] = df_all['RoadType'].astype(str) + "_" + df_all['LargeVehicles'].astype(str)
    df_all['Road_Weather'] = df_all['RoadType'].astype(str) + "_" + df_all['Weather'].astype(str)
    
    train_feat = df_all[df_all['is_test'] == 0].drop(columns=['is_test', 'demand'])
    train_feat['demand'] = train['demand'].values 
    test_feat = df_all[df_all['is_test'] == 1].drop(columns=['is_test', 'demand'])
    
    return train_feat, test_feat

train_feat, test_feat = engineer_mega_features(train_raw, test_raw)

TE_COLS = ['geohash', 'geo_prefix4', 'geo_prefix5', 'RoadType', 'Weather', 'Road_Weather', 'Road_LV', 'LargeVehicles']
DROP_BASE = ['Index','demand','geohash','timestamp','RoadType','Weather','LargeVehicles','Landmarks','geo_prefix4','geo_prefix5', 'Road_Weather', 'Road_LV', 'is_weekend', 'RoadType_Clean', 'is_non_residential']
BASE_COLS = [c for c in train_feat.columns if c not in DROP_BASE and c != 'demand']
BASE_COLS.extend(['lag_gh_hour', 'hourly_growth_ratio', 'weather_growth_ratio', 'expected_demand_adjusted'])
print("Feature engineering complete!")

--- 🚀 ENGINEERING CLEAN MEGA FEATURES ---
Feature engineering complete!


In [19]:
print("--- ⏱️ CLUSTERING, DYNAMIC ANCHORING & IMPUTATION ---")

# 🛠️ THE FIX: Force 'demand' and 'day' to be numeric immediately!
train_feat['demand'] = pd.to_numeric(train_feat['demand'], errors='coerce')

# ── 1. Geographic Clustering ──
valid_coords = train_feat[['latitude', 'longitude']].dropna().drop_duplicates()
kmeans = KMeans(n_clusters=20, random_state=SEED, n_init=10)
kmeans.fit(valid_coords)

def assign_clusters(df, kmeans_model):
    clusters = np.full(len(df), -1) 
    mask = df['latitude'].notna()
    if mask.sum() > 0:
        clusters[mask] = kmeans_model.predict(df.loc[mask, ['latitude', 'longitude']])
    return clusters.astype(str) 

train_feat['geo_cluster'] = assign_clusters(train_feat, kmeans)
test_feat['geo_cluster']  = assign_clusters(test_feat, kmeans)
if 'geo_cluster' not in TE_COLS: TE_COLS.append('geo_cluster')

# ── 2. Dynamic Time Anchoring ──
for df in [train_feat, test_feat]:
    if 'day' in df.columns:
        df['day'] = pd.to_numeric(df['day'], errors='coerce').fillna(0).astype(int)

if 'day' not in train_feat.columns:
    train_feat['day'] = train_feat.index // (max(1, len(train_feat) // 48))

LAST_TRAIN_DAY = train_feat['day'].max()
PREV_TRAIN_DAY = LAST_TRAIN_DAY - 1

print(f"Anchoring momentum dynamically to Day {LAST_TRAIN_DAY} and Day {PREV_TRAIN_DAY}...")

day_current_df = train_feat[train_feat['day'] == LAST_TRAIN_DAY]
day_prev_df = train_feat[train_feat['day'] == PREV_TRAIN_DAY]

gh_hr_anchor_mean = day_current_df.groupby(['geohash', 'hour'])['demand'].mean().to_dict()
reg_hr_anchor_mean = day_current_df.groupby(['geo_prefix5', 'hour'])['demand'].mean().to_dict()
global_anchor = day_current_df['demand'].mean()

hr_prev_mean = day_prev_df.groupby('hour')['demand'].mean()
hr_curr_mean = day_current_df.groupby('hour')['demand'].mean()
hourly_ratio = (hr_curr_mean / hr_prev_mean).fillna(1.0).to_dict()

wea_prev_mean = day_prev_df.groupby('Weather')['demand'].mean()
wea_curr_mean = day_current_df.groupby('Weather')['demand'].mean()
weather_ratio = (wea_curr_mean / wea_prev_mean).fillna(1.0).to_dict()

for df in [train_feat, test_feat]:
    df['gh_hour_idx'] = list(zip(df['geohash'], df['hour']))
    df['reg_hour_idx'] = list(zip(df['geo_prefix5'], df['hour']))
    
    df['lag_gh_hour'] = df['gh_hour_idx'].map(gh_hr_anchor_mean)
    reg_fallback = df['reg_hour_idx'].map(reg_hr_anchor_mean)
    df['lag_gh_hour'] = df['lag_gh_hour'].fillna(reg_fallback).fillna(global_anchor)
    
    df['expected_demand_adjusted'] = df['lag_gh_hour'] * df['hour'].map(hourly_ratio).fillna(1.0) * df['Weather'].map(weather_ratio).fillna(1.0)

# ── 3. Memory-Safe Imputation ──
print("Running Sklearn Imputers...")
dynamic_base_cols = [c for c in BASE_COLS if c in train_feat.columns]
if 'expected_demand_adjusted' not in dynamic_base_cols: dynamic_base_cols.append('expected_demand_adjusted')

num_imp = KNNImputer(n_neighbors=5, weights='distance')
Xtr_base = pd.DataFrame(num_imp.fit_transform(train_feat[dynamic_base_cols]), columns=dynamic_base_cols, index=train_feat.index)
Xte_base = pd.DataFrame(num_imp.transform(test_feat[dynamic_base_cols]), columns=dynamic_base_cols, index=test_feat.index)

cat_imp = SimpleImputer(strategy='constant', fill_value='Missing')
Xtr_cats = pd.DataFrame(cat_imp.fit_transform(train_feat[TE_COLS]), columns=TE_COLS, index=train_feat.index)
Xte_cats = pd.DataFrame(cat_imp.transform(test_feat[TE_COLS]), columns=TE_COLS, index=test_feat.index)


# ── [UPDATED SECTION AT END OF CELL 4] ──

X_train_full = pd.concat([Xtr_base, Xtr_cats], axis=1)
X_test_full = pd.concat([Xte_base, Xte_cats], axis=1)

# 🛠️ THE FIX: Handle any lingering NaNs in the target variable
train_feat['demand'] = train_feat['demand'].fillna(train_feat['demand'].median())

# Now calculate log1p safely
y_train_full = np.log1p(train_feat['demand'].values)

# ── [CONTINUE TO GPU SETUP SECTION] ──

# ── 4. GPU Setup (Label Encoding LightGBM) ──
print("Encoding Categories for GPU compatibility...")
X_train_lgb = X_train_full.copy(); X_test_lgb = X_test_full.copy()
X_train_cat = X_train_full.copy(); X_test_cat = X_test_full.copy()

for c in TE_COLS:
    X_train_cat[c] = X_train_cat[c].astype(str)
    X_test_cat[c] = X_test_cat[c].astype(str)
    
    le = LabelEncoder()
    full_col = pd.concat([X_train_lgb[c], X_test_lgb[c]]).astype(str)
    le.fit(full_col)
    X_train_lgb[c] = le.transform(X_train_lgb[c].astype(str))
    X_test_lgb[c] = le.transform(X_test_lgb[c].astype(str))

--- ⏱️ CLUSTERING, DYNAMIC ANCHORING & IMPUTATION ---
Anchoring momentum dynamically to Day 49 and Day 48...
Running Sklearn Imputers...
Encoding Categories for GPU compatibility...


In [20]:
print("\n--- 🏆 RUNNING THE MEGA K-FOLD ENSEMBLE (ON GPU) ---")

N_FOLDS = 5
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# GPU Flags Activated
lgb_p = dict(n_estimators=3500, learning_rate=0.015, num_leaves=110, min_child_samples=25, 
             subsample=0.8, colsample_bytree=0.8, reg_alpha=0.5, reg_lambda=3.0, 
             random_state=SEED, n_jobs=-1, verbosity=-1, device='gpu')

# 🚀 FIXED cat_p DICTIONARY
cat_p = dict(
    iterations=3500, 
    learning_rate=0.02, 
    depth=7, 
    l2_leaf_reg=5, 
    bootstrap_type='Bernoulli', # <--- THIS IS THE FIX
    subsample=0.8, 
    min_data_in_leaf=30, 
    random_seed=SEED, 
    early_stopping_rounds=150, 
    verbose=False, 
    cat_features=TE_COLS, 
    task_type='GPU'
)

final_test_preds = np.zeros(len(X_test_full))
oof_preds = np.zeros(len(X_train_full))

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_full), 1):
    print(f"\n🔄 Training Fold {fold}...")
    
    X_tr_lgb, y_tr = X_train_lgb.iloc[tr_idx], y_train_full[tr_idx]
    X_va_lgb, y_va = X_train_lgb.iloc[va_idx], y_train_full[va_idx]
    X_tr_cat, X_va_cat = X_train_cat.iloc[tr_idx], X_train_cat.iloc[va_idx]
    
    # 1. Train LightGBM 
    m_lgb = lgb.LGBMRegressor(**lgb_p)
    m_lgb.fit(X_tr_lgb, y_tr, eval_set=[(X_va_lgb, y_va)], callbacks=[lgb.early_stopping(150, verbose=False)])
    
    # 2. Train CatBoost
    m_cat = CatBoostRegressor(**cat_p)
    m_cat.fit(X_tr_cat, y_tr, eval_set=(X_va_cat, y_va))
    
    # 3. SCORE VALIDATION FOLD
    va_preds_lgb = m_lgb.predict(X_va_lgb)
    va_preds_cat = m_cat.predict(X_va_cat)
    va_blend = (va_preds_lgb * 0.6) + (va_preds_cat * 0.4)
    oof_preds[va_idx] = va_blend
    print(f"✅ Fold {fold} OOF R² Score: {r2_score(y_va, va_blend):.5f}")
    
    # 4. Predict on Test Set
    fold_preds_lgb = np.expm1(m_lgb.predict(X_test_lgb))
    fold_preds_cat = np.expm1(m_cat.predict(X_test_cat))
    fold_blend_test = (fold_preds_lgb * 0.6) + (fold_preds_cat * 0.4)
    final_test_preds += fold_blend_test / N_FOLDS

print("\n" + "="*40)
print(f"🏆 OVERALL TRUTH ENSEMBLE R² : {r2_score(y_train_full, oof_preds):.5f}")
print("="*40)

# Generate Submission
sub_out = pd.DataFrame({
    'Index': test_raw['Index'].values,
    'demand': np.clip(final_test_preds, 0, None)
})
sub_out.to_csv('submission_FINAL_CLEAN.csv', index=False)
print("✅ 'submission_FINAL_CLEAN.csv' written successfully!")


--- 🏆 RUNNING THE MEGA K-FOLD ENSEMBLE (ON GPU) ---

🔄 Training Fold 1...
✅ Fold 1 OOF R² Score: 0.97153

🔄 Training Fold 2...
✅ Fold 2 OOF R² Score: 0.97316

🔄 Training Fold 3...
✅ Fold 3 OOF R² Score: 0.97271

🔄 Training Fold 4...
✅ Fold 4 OOF R² Score: 0.97309

🔄 Training Fold 5...
✅ Fold 5 OOF R² Score: 0.97306

🏆 OVERALL TRUTH ENSEMBLE R² : 0.97271
✅ 'submission_FINAL_CLEAN.csv' written successfully!
